In [1]:
# ruff: noqa: F405
%cd /lustre/scratch126/gengen/teams_v2/marks/dp31/elia/spatialPeeler/demo

from _shared import pl_k_sP, psc_ds, run_deg_groups, run_nmf, run_proc_per_ds, viz_defs, comp_spat_stats  # isort:skip

from functools import partial
from pathlib import Path
from pickle import dump, load
from warnings import catch_warnings, simplefilter

import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
import uapl
import upsetplot as up
from matplotlib import pyplot as plt
from plotnine import *  # noqa: F403

import spatialpeeler as sP
from spatialpeeler.diag import comp_sparsity, sweep_uns

/lustre/scratch126/gengen/teams_v2/marks/dp31/elia/spatialPeeler/demo


/lustre/scratch126/gengen/teams_v2/marks/dp31/elia/spatialPeeler/.venv/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [ ]:
viz_defs(dpi=128)
RAND_SEED = 28
# define helper function for plotting
pl_help = partial(pl_k_sP, cond_col="group", sample_col="sample_id", ctl_val="CTL")
SP_FACT_PAT_STR = "sP_p_hat_k={k}"
SP_GENE_PAT_STR = "sp_gene_id_k={k}"

CASE_KS = [4, 11, 7, 15, 1, 24, 13, 22, 27]
CTL_KS = [6, 8, 5, 10, 2, 9, 0, 12, 17, 24]

CASE_KS = [4, 24, 27]
CTL_KS= []

In [3]:
# load dataset + run NMF
if (pth := Path("../.cache/psc.pkl")).exists():
    with open(pth, "rb") as f:
        psc = load(f)
else:
    adatas = psc_ds(
        Path("../data/psc-visium"),
    )
    for k in adatas:
        adatas[k] = adatas[k][adatas[k].obs["in_tissue"], :]

    run_proc_per_ds(adatas)

    psc = ad.concat(adatas, label="sample_id", index_unique="_")

    sc.pp.filter_genes(psc, min_cells=psc.n_obs // 500)

    psc.var["SYMBOL"] = (
        sc.queries.biomart_annotations("hsapiens", ["ensembl_gene_id", "external_gene_name"], host="grch37.ensembl.org")
        .set_index("ensembl_gene_id")
        .merge(psc.var.index.to_series(), left_index=True, right_index=True, how="right")["external_gene_name"]
        .fillna(psc.var.index.to_series())
    )

    run_nmf(psc, 30, RAND_SEED)

    pth.parent.mkdir(exist_ok=True)
    with open(pth, "wb") as f:
        dump(psc, f)
print(psc)

AnnData object with n_obs × n_vars = 21553 × 16465
    obs: 'in_tissue', 'percent.mt', 'cell_type', 'group', 'sample_id'
    var: 'n_cells', 'SYMBOL'
    obsm: 'spatial', 'X_nmf'
    varm: 'X_nmf'
    layers: 'counts'


In [4]:
from IPython.display import display
import plotnine as p9
import uapl

# compute log-normalized expression from raw counts, store as a new layer
tmp = psc.copy()
tmp.X = tmp.layers["counts"]
sc.pp.normalize_total(tmp, target_sum=1e4)
sc.pp.log1p(tmp)
psc.layers["lognorm"] = tmp.X

In [ ]:
genelist = ['KRT7', 'IL32', 'AKR1B10', 'SQSTM1'] #'CXCL8', 
pericentral = ['GLUL', 'CYP2E1', 'CYP3A4']
periportal = ['CYP2A6', 'CYP2A7', 'APOA1', 'ALDOB']
mild_periportal = ['TTR', 'HP']

genelist =  mild_periportal
for gene in genelist:
    gene_id = psc.var.index[psc.var["SYMBOL"] == gene][0]

    p = (
        uapl.PlotAdata(psc)
        + p9.aes(x=uapl.obsm("spatial", "x"), y=uapl.obsm("spatial", "y"), color=uapl.X("lognorm", gene_id))
        + p9.geom_point(shape="h", size=1.4, stroke=0)
        + p9.coord_fixed()
        + p9.facet_wrap(uapl.obs("sample_id"), scales="free", ncol=4)

        + p9.scale_color_cmap("viridis")
        + p9.theme(axis_text=p9.element_blank(), axis_ticks=p9.element_blank())
        + p9.labs(color=gene, title=f"{gene} normalized expression across samples")
    )
    display(p)

    

In [ ]:

gene = "MYH11"
gene_id = psc.var.index[psc.var["SYMBOL"] == gene][0]

(
    uapl.PlotAdata(psc)
    + p9.aes(x=uapl.obsm("spatial", "x"), y=uapl.obsm("spatial", "y"), color=uapl.X("lognorm", gene_id))
    + p9.geom_point(shape="h", size=1.4, stroke=0)
    + p9.coord_fixed()
    + p9.facet_wrap(uapl.obs("sample_id"), scales="free", ncol=4)
    + p9.scale_color_cmap("viridis")
    + p9.theme(axis_text=p9.element_blank(), axis_ticks=p9.element_blank())
    + p9.labs(color=gene, title=f"{gene} normalized expression across samples")
)
